**Build a chatbot that:**  

* Answers questions from your documents.
* Remembers previous conversation (chat history).
* Responds more naturally like ChatGPT.

In [1]:
from langchain_community.document_loaders import PyPDFLoader

from langchain_text_splitters import RecursiveCharacterTextSplitter

from langchain_huggingface import HuggingFaceEmbeddings

from langchain_community.vectorstores import FAISS

from google import genai

import os

C:\Users\Dell\AppData\Local\Temp\ipykernel_6616\2954837846.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


**Load Multiple PDFs**

In [2]:
folder = "day16_pdfs"

documents = []

for file in os.listdir (folder):
    if file.endswith(".pdf"):
        loader = PyPDFLoader(os.path.join(folder, file))
        documents.extend(loader.load())

print("Pages Loade:", len(documents))

Pages Loade: 121


**check metadat**

In [3]:
print(documents[0].metadata)

{'producer': 'pdfTeX-1.40.25', 'creator': 'LaTeX with hyperref', 'creationdate': '2024-04-10T21:11:43+00:00', 'author': '', 'keywords': '', 'moddate': '2024-04-10T21:11:43+00:00', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.25 (TeX Live 2023) kpathsea version 6.3.5', 'subject': '', 'title': '', 'trapped': '/False', 'source': 'day16_pdfs\\attension-is-all-u-need.pdf', 'total_pages': 15, 'page': 0, 'page_label': '1'}


**Split Documents**

In [4]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size = 500,
    chunk_overlap = 100
)

chunks = splitter.split_documents(documents)

print("Total chunks:", len(chunks))

Total chunks: 863


**Create Embeddings**

In [5]:
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

**Store in FAISS**

In [6]:
vector_db = FAISS.from_documents(
    chunks,
    embeddings
)
print("vector database ready")

vector database ready


**Create Retriever**

In [7]:
retriever = vector_db.as_retriever(
    search_kwargs={"k":3}
)
print("retriever is ready")

retriever is ready


**Create Chat History**

In [8]:
chat_history = []

**user Query**

In [10]:
query = "What is Transformer?"

**Retrieve Relevant Documents**

In [11]:
retrieved_docs = retriever.invoke(query)

for i, doc in enumerate(retrieved_docs):
    print("="*60)
    print(f"Document {i+1}")
    print(doc.page_content)

Document 1
Figure 1: The Transformer - model architecture.
The Transformer follows this overall architecture using stacked self-attention and point-wise, fully
connected layers for both the encoder and decoder, shown in the left and right halves of Figure 1,
respectively.
3.1 Encoder and Decoder Stacks
Encoder: The encoder is composed of a stack of N = 6 identical layers. Each layer has two
sub-layers. The first is a multi-head self-attention mechanism, and the second is a simple, position-
Document 2
The Transformer uses multi-head attention in three different ways:
• In "encoder-decoder attention" layers, the queries come from the previous decoder layer,
and the memory keys and values come from the output of the encoder. This allows every
position in the decoder to attend over all positions in the input sequence. This mimics the
typical encoder-decoder attention mechanisms in sequence-to-sequence models such as
[38, 2, 9].
Document 3
ture and the ﬁnal downstream architecture.
Model A

**Create Context**

In [12]:
context = "\n\n".join(
    [doc.page_content for doc in retrieved_docs]
)

print(context)

Figure 1: The Transformer - model architecture.
The Transformer follows this overall architecture using stacked self-attention and point-wise, fully
connected layers for both the encoder and decoder, shown in the left and right halves of Figure 1,
respectively.
3.1 Encoder and Decoder Stacks
Encoder: The encoder is composed of a stack of N = 6 identical layers. Each layer has two
sub-layers. The first is a multi-head self-attention mechanism, and the second is a simple, position-

The Transformer uses multi-head attention in three different ways:
• In "encoder-decoder attention" layers, the queries come from the previous decoder layer,
and the memory keys and values come from the output of the encoder. This allows every
position in the decoder to attend over all positions in the input sequence. This mimics the
typical encoder-decoder attention mechanisms in sequence-to-sequence models such as
[38, 2, 9].

ture and the ﬁnal downstream architecture.
Model Architecture BERT’s model archit

**Build Prompt**

In [13]:
prompt = f"""
You are an AI assistant.

Previous Conversation:
{chat_history}

Answer ONLY using the context below.

If the answer is not available, reply:
"I don't know based on the provided context."

Context:
{context}

Question:
{query}

Answer:
"""

print(prompt)


You are an AI assistant.

Previous Conversation:
[]

Answer ONLY using the context below.

If the answer is not available, reply:
"I don't know based on the provided context."

Context:
Figure 1: The Transformer - model architecture.
The Transformer follows this overall architecture using stacked self-attention and point-wise, fully
connected layers for both the encoder and decoder, shown in the left and right halves of Figure 1,
respectively.
3.1 Encoder and Decoder Stacks
Encoder: The encoder is composed of a stack of N = 6 identical layers. Each layer has two
sub-layers. The first is a multi-head self-attention mechanism, and the second is a simple, position-

The Transformer uses multi-head attention in three different ways:
• In "encoder-decoder attention" layers, the queries come from the previous decoder layer,
and the memory keys and values come from the output of the encoder. This allows every
position in the decoder to attend over all positions in the input sequence. This mi

**Send Prompt to Gemini**

In [14]:
client = genai.Client(api_key="Your_API_KEY")

response = client.models.generate_content(
    model="gemini-2.5-flash",
    contents=prompt
)

answer = response.text

print(answer)

The Transformer is a model architecture that uses stacked self-attention and point-wise, fully connected layers for both the encoder and decoder.


**Save Conversation**

In [15]:
chat_history += f"\nUser: {query}"
chat_history += f"\nAssistant: {answer}"

print(chat_history)

['\n', 'U', 's', 'e', 'r', ':', ' ', 'W', 'h', 'a', 't', ' ', 'i', 's', ' ', 'T', 'r', 'a', 'n', 's', 'f', 'o', 'r', 'm', 'e', 'r', '?', '\n', 'A', 's', 's', 'i', 's', 't', 'a', 'n', 't', ':', ' ', 'T', 'h', 'e', ' ', 'T', 'r', 'a', 'n', 's', 'f', 'o', 'r', 'm', 'e', 'r', ' ', 'i', 's', ' ', 'a', ' ', 'm', 'o', 'd', 'e', 'l', ' ', 'a', 'r', 'c', 'h', 'i', 't', 'e', 'c', 't', 'u', 'r', 'e', ' ', 't', 'h', 'a', 't', ' ', 'u', 's', 'e', 's', ' ', 's', 't', 'a', 'c', 'k', 'e', 'd', ' ', 's', 'e', 'l', 'f', '-', 'a', 't', 't', 'e', 'n', 't', 'i', 'o', 'n', ' ', 'a', 'n', 'd', ' ', 'p', 'o', 'i', 'n', 't', '-', 'w', 'i', 's', 'e', ',', ' ', 'f', 'u', 'l', 'l', 'y', ' ', 'c', 'o', 'n', 'n', 'e', 'c', 't', 'e', 'd', ' ', 'l', 'a', 'y', 'e', 'r', 's', ' ', 'f', 'o', 'r', ' ', 'b', 'o', 't', 'h', ' ', 't', 'h', 'e', ' ', 'e', 'n', 'c', 'o', 'd', 'e', 'r', ' ', 'a', 'n', 'd', ' ', 'd', 'e', 'c', 'o', 'd', 'e', 'r', '.']


**Follow-up Question**

In [16]:
query = "Can you explain it in simple words?"

print(query)

Can you explain it in simple words?


**Retrieve Documents Again**

In [17]:
retrieved_docs = retriever.invoke(query)

context = "\n\n".join(
    [doc.page_content for doc in retrieved_docs]
)

print(context)

This paragraph provides a brief overview of the flat Earth
theory. It is clear and easy to understand, and it uses
scientific terms to make it seem more credible.
> **Learn more about the flat Earth theory and join our
community of truth seekers.**
This call to action is clear and concise, and it encourages
visitors to take action. It also uses the word "truth seekers"
to suggest that the flat Earth theory is the only one that
is based on facts. I hope these ideas help you create a

step-by-step explanation and correctly defined LaTeX equations.
Source: question is provided by Macmillan Learning.
87

Figure 18| Common-sense reasoning in images. The model is able to understand the relationships
represented in the graphs and reason about them in a multilingual setting.
Source: image created by an author from the Gemini team.
84


**Build a New Prompt**

In [35]:
prompt = f"""
You are an AI assistant.

Previous Conversation:
{chat_history}

Answer ONLY using the context below.

If the answer is not available, reply:
"I don't know based on the provided context."

Context:
{context}

Question:
{query}

Answer:
"""

print(prompt)


You are an AI assistant.

Previous Conversation:
['\n', 'U', 's', 'e', 'r', ':', ' ', 'W', 'h', 'a', 't', ' ', 'i', 's', ' ', 'T', 'r', 'a', 'n', 's', 'f', 'o', 'r', 'm', 'e', 'r', '?', '\n', 'A', 's', 's', 'i', 's', 't', 'a', 'n', 't', ':', ' ', 'T', 'h', 'e', ' ', 'T', 'r', 'a', 'n', 's', 'f', 'o', 'r', 'm', 'e', 'r', ' ', 'i', 's', ' ', 'a', ' ', 'm', 'o', 'd', 'e', 'l', ' ', 'a', 'r', 'c', 'h', 'i', 't', 'e', 'c', 't', 'u', 'r', 'e', ' ', 't', 'h', 'a', 't', ' ', 'u', 's', 'e', 's', ' ', 's', 't', 'a', 'c', 'k', 'e', 'd', ' ', 's', 'e', 'l', 'f', '-', 'a', 't', 't', 'e', 'n', 't', 'i', 'o', 'n', ' ', 'a', 'n', 'd', ' ', 'p', 'o', 'i', 'n', 't', '-', 'w', 'i', 's', 'e', ',', ' ', 'f', 'u', 'l', 'l', 'y', ' ', 'c', 'o', 'n', 'n', 'e', 'c', 't', 'e', 'd', ' ', 'l', 'a', 'y', 'e', 'r', 's', ' ', 'f', 'o', 'r', ' ', 'b', 'o', 't', 'h', ' ', 't', 'h', 'e', ' ', 'e', 'n', 'c', 'o', 'd', 'e', 'r', ' ', 'a', 'n', 'd', ' ', 'd', 'e', 'c', 'o', 'd', 'e', 'r', '.']

Answer ONLY using the cont

**Generate Answer**

In [36]:
response = client.models.generate_content(
    model="gemini-2.5-flash",
    contents=prompt
)

answer = response.text

print(answer)

I don't know based on the provided context.


**Update Chat History Again**

In [37]:
chat_history += f"\nUser: {query}"
chat_history += f"\nAssistant: {answer}"

print(chat_history)

['\n', 'U', 's', 'e', 'r', ':', ' ', 'W', 'h', 'a', 't', ' ', 'i', 's', ' ', 'T', 'r', 'a', 'n', 's', 'f', 'o', 'r', 'm', 'e', 'r', '?', '\n', 'A', 's', 's', 'i', 's', 't', 'a', 'n', 't', ':', ' ', 'T', 'h', 'e', ' ', 'T', 'r', 'a', 'n', 's', 'f', 'o', 'r', 'm', 'e', 'r', ' ', 'i', 's', ' ', 'a', ' ', 'm', 'o', 'd', 'e', 'l', ' ', 'a', 'r', 'c', 'h', 'i', 't', 'e', 'c', 't', 'u', 'r', 'e', ' ', 't', 'h', 'a', 't', ' ', 'u', 's', 'e', 's', ' ', 's', 't', 'a', 'c', 'k', 'e', 'd', ' ', 's', 'e', 'l', 'f', '-', 'a', 't', 't', 'e', 'n', 't', 'i', 'o', 'n', ' ', 'a', 'n', 'd', ' ', 'p', 'o', 'i', 'n', 't', '-', 'w', 'i', 's', 'e', ',', ' ', 'f', 'u', 'l', 'l', 'y', ' ', 'c', 'o', 'n', 'n', 'e', 'c', 't', 'e', 'd', ' ', 'l', 'a', 'y', 'e', 'r', 's', ' ', 'f', 'o', 'r', ' ', 'b', 'o', 't', 'h', ' ', 't', 'h', 'e', ' ', 'e', 'n', 'c', 'o', 'd', 'e', 'r', ' ', 'a', 'n', 'd', ' ', 'd', 'e', 'c', 'o', 'd', 'e', 'r', '.', '\n', 'U', 's', 'e', 'r', ':', ' ', 'C', 'a', 'n', ' ', 'y', 'o', 'u', ' ', '

**Create the Function**

In [18]:
def ask_question(query):

    global chat_history

    # Retrieve relevant documents
    retrieved_docs = retriever.invoke(query)

    # Create context
    context = "\n\n".join(
        [doc.page_content for doc in retrieved_docs]
    )

    # Build prompt
    prompt = f"""
You are an AI assistant.

Rules:
1. Answer ONLY using the provided context.
2. If the answer is not found in the context, reply exactly:
   "I don't know based on the provided context."
3. Keep the answer short and concise (maximum 2-3 sentences).
4. Do NOT copy long paragraphs from the context.
5. Answer only the user's question.
6. If the question asks "Who", return only the person's or organization's name.
7. If the question asks "What", provide only the definition or explanation.
8. If the question asks "When", provide only the date or time.
9. If the question asks "Where", provide only the location.
10. Do not include extra details unless the user specifically asks for them.

Previous Conversation:
{chat_history}

Answer ONLY using the context below.

If the answer is not available, reply:
"I don't know based on the provided context."

Context:
{context}

Question:
{query}

Answer:
"""

    # Send prompt to Gemini
    response = client.models.generate_content(
        model="gemini-2.5-flash",
        contents=prompt
    )

    answer = response.text

    # Update chat history
    chat_history += f"\nUser: {query}"
    chat_history += f"\nAssistant: {answer}\n"

    return answer

In [40]:
print(ask_question("What is the Transformer architecture?"))

The Transformer architecture uses stacked self-attention and point-wise, fully connected layers for both the encoder and decoder.


**Build the Final Chatbot**

In [19]:
while True:

    query = input("\nYou: ")

    if query.lower() == "exit":
        print("Goodbye!")
        break

    answer = ask_question(query)

    print("\nAssistant:")
    print(answer)


You:  tell me the fine  tunig in bert



Assistant:
I don't know based on the provided context.



You:  tell me the fine tuning in bert



Assistant:
Fine-tuning in BERT is straightforward, leveraging the Transformer's self-attention mechanism to model various downstream tasks by swapping appropriate inputs and outputs. BERT selects a task-specific fine-tuning learning rate that performs best on the development set.



You:  performance of gemini ultra


...


Assistant:
I don't know based on the provided context.



You:  shahid



Assistant:
I don't know based on the provided context.



You:  exit


Goodbye!
